# Bronze Layer

## Source Ingestion

Purpose:
- Read the raw landing file a PostgresExtractionStage produced, per entity
- Persist it as Delta in the Bronze layer
- Validate the result

## Environment Bootstrap

Dependencies (the scoped `modern-data-platform` wheel + `pydantic`) come from the `mdp-bronze-silver` workspace base environment, referenced by this job's `environment_key` (`infrastructure/databricks/*_job.yml`) -- not a per-notebook `%pip install` cell anymore. For serverless notebook tasks, a job-level environment overrides whatever's selected in this notebook's own Environment side panel, so no manual UI step is needed either.

See `docs/architecture/roadmap-next-steps.md` for why this replaced the old `%pip install $wheel_glob` + `restartPython()` cell (found to be reinstalling the *entire* monorepo's dependencies, including apache-airflow, on every one of the 4 pipeline stages), the portability trade-off (the environment's wheel path is tied to one real deployment, not templated per-target like `wheel_path` used to be), and the stale-cache risk this now carries (the wheel's filename doesn't change per deploy -- a `refresh-workspace-base-environment` call is needed after `databricks bundle deploy` if the wheel's content changed but its version didn't).

## Imports

In [ ]:
from data_platform.compute.delta_io import read_delta, read_raw, write_delta
from data_platform.compute.spark import get_spark
from data_platform.storage.config import StorageConfig
from integrations.databricks.runtime.parameters import get_parameter

## Parameters

`entities` is a comma-separated list (e.g. `customers,orders,products`), parsed here -- a single value still works (`"customers".split(",")` == `["customers"]`). Processing multiple entities in one notebook execution means the wheel install and SparkSession above are paid once per run, not once per entity.

In [ ]:
entities = [
    entity.strip()
    for entity in get_parameter("entities", default="customers").split(",")
    if entity.strip()
]

## Spark Session

Created once, reused across every entity in the loop below.

In [ ]:
spark = get_spark("Bronze Ingestion")

## Ingest Function

In [ ]:
def ingest_entity(spark, entity: str) -> None:
    raw_df = read_raw(spark, StorageConfig.raw(entity))

    raw_df.printSchema()
    raw_df.show(10)

    write_delta(raw_df, StorageConfig.bronze_batch(entity), mode="overwrite")

    bronze_df = read_delta(spark, StorageConfig.bronze_batch(entity))

    print(f"[{entity}] Total records: {bronze_df.count()}")

## Run For Each Entity

Entities are independent of each other, so one failing must not cost reprocessing the others: each is wrapped in its own try/except, errors are collected, and only surface as a single aggregated failure at the end -- which still fails the Databricks task for real (no silent partial success).

In [ ]:
errors: dict[str, str] = {}

for entity in entities:
    print(f"=== {entity}: ingesting ===")

    try:
        ingest_entity(spark, entity)

    except Exception as error:
        print(f"[{entity}] FAILED: {error}")
        errors[entity] = str(error)

if errors:
    summary = "\n".join(
        f"  - {entity}: {message}" for entity, message in errors.items()
    )

    raise RuntimeError(
        f"{len(errors)} of {len(entities)} entities failed:\n{summary}"
    )

print(f"OK: {len(entities)} entities processed: {', '.join(entities)}")